In [1]:
import re
import html
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# PREPROCESS
# ==========================================
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = html.unescape(text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ==========================================
# LOAD MODEL
# ==========================================
model_path = "./indobert_smote70"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

model.to(device)
model.eval()

# ==========================================
# PREDICT FUNCTION
# ==========================================
def predict(text):

    text = preprocess_text(text)

    encoding = tokenizer(
        text,
        max_length=128,
        truncation=True,
        padding='max_length',
        return_tensors='pt'
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        prob = torch.sigmoid(outputs.logits).item()

    label = 1 if prob >= 0.5 else 0

    return label, prob

# ==========================================
# TEST
# ==========================================
text = "Aplikasi ini sangat membantu dan mudah digunakan"

label, prob = predict(text)

print("Prediksi :", label)
print("Probabilitas positif :", round(prob, 4))

c:\SKRIPSI BRYAN\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 869.17it/s, Materializing param=classifier.weight]                                      


Prediksi : 0
Probabilitas positif : 0.3564
